# EXERCISE 5: TEXT CLASSIFICATION USING NAÏVE BAYES AND SVM
### Aim:
To design, train, and evaluate machine learning text classification models:
1. **Spam Email Detection** using **Multinomial Naïve Bayes**
2. **Sentiment Analysis** (Multi-class: Positive, Negative, Neutral) using **Support Vector Machine (LinearSVC)**
3. **Multi-topic Document Classification** (Post-Lab)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries imported successfully.")

Libraries imported successfully.


## PART 1: SPAM EMAIL DETECTION USING NAÏVE BAYES

In [2]:
# Spam Email Dataset
spam_emails = [
    # 10 Spam emails
    "Urgent account update: Claim your $500 gift card reward now",
    "Exclusive crypto offer: Double your bitcoin investment today",
    "You have won the international grand prize sweepstakes",
    "Claim your complimentary luxury cruise holiday today",
    "Immediate cash payout waiting: Verify your details immediately",
    "Special promotional discount: Claim 90% off your purchase now",
    "Congratulations winner! Collect your complimentary smartphone today",
    "Risk-free loan approved: Claim your cash transfer immediately",
    "Free trial access granted: Claim your bonus credits now",
    "Hot investment alert: Guaranteed high returns click here",

    # 10 Not Spam emails
    "Please find the quarterly financial budget report attached",
    "The department colloquium is scheduled for Wednesday at 10 AM",
    "Kindly submit the completed lab assignments by Friday afternoon",
    "Team standup meeting notes and action items from yesterday",
    "Your flight ticket confirmation for the upcoming tech conference",
    "Please review the updated project architecture diagram before the call",
    "Semester exam schedule and hall ticket details are announced",
    "Code review requested for pull request 42 on github",
    "Faculty advisor office hours are postponed to Thursday morning",
    "Monthly cloud infrastructure usage summary and metrics report"
]

spam_labels = ["Spam"] * 10 + ["Not Spam"] * 10

spam_df = pd.DataFrame({
    "email": spam_emails,
    "label": spam_labels
})

print("Dataset preview:")
print(spam_df.head(6))

Dataset preview:
                                               email label
0  Urgent account update: Claim your $500 gift ca...  Spam
1  Exclusive crypto offer: Double your bitcoin in...  Spam
2  You have won the international grand prize swe...  Spam
3  Claim your complimentary luxury cruise holiday...  Spam
4  Immediate cash payout waiting: Verify your det...  Spam
5  Special promotional discount: Claim 90% off yo...  Spam


In [3]:
print("Dataset shape:", spam_df.shape)
print("\nClass distribution:")
print(spam_df["label"].value_counts())

Dataset shape: (20, 2)

Class distribution:
label
Spam        10
Not Spam    10
Name: count, dtype: int64


In [4]:
X_text = spam_df["email"]
y = spam_df["label"]

tfidf_spam = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

X_tfidf = tfidf_spam.fit_transform(X_text)

print("TF-IDF matrix shape:", X_tfidf.shape)
print("Vocabulary size:", len(tfidf_spam.vocabulary_))

TF-IDF matrix shape: (20, 225)
Vocabulary size: 225


In [5]:
feature_names = tfidf_spam.get_feature_names_out()

print("First 20 TF-IDF features:")
print(feature_names[:20])

First 20 TF-IDF features:
['10' '42' '42 github' '500' '500 gift' '90' '90 purchase' 'access'
 'access granted' 'account' 'account update' 'action' 'action items'
 'advisor' 'advisor office' 'afternoon' 'alert' 'alert guaranteed'
 'announced' 'approved']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Training samples :", X_train.shape[0])
print("Testing samples  :", X_test.shape[0])

Training samples : 14
Testing samples  : 6


In [7]:
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train, y_train)

print("Multinomial Naïve Bayes model trained successfully.")

Multinomial Naïve Bayes model trained successfully.


In [8]:
y_pred_nb = nb_model.predict(X_test)

comparison_nb = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred_nb
})

print("Naïve Bayes Predictions on Test Set:")
print(comparison_nb)

Naïve Bayes Predictions on Test Set:
     Actual Predicted
0  Not Spam  Not Spam
1      Spam      Spam
2  Not Spam  Not Spam
3      Spam      Spam
4      Spam  Not Spam
5  Not Spam      Spam


In [9]:
accuracy_nb = accuracy_score(y_test, y_pred_nb)
print(f"Naïve Bayes Accuracy: {accuracy_nb * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

Naïve Bayes Accuracy: 66.67%

Classification Report:
              precision    recall  f1-score   support

    Not Spam       0.67      0.67      0.67         3
        Spam       0.67      0.67      0.67         3

    accuracy                           0.67         6
   macro avg       0.67      0.67      0.67         6
weighted avg       0.67      0.67      0.67         6



In [10]:
labels_nb = ["Not Spam", "Spam"]
cm_nb = confusion_matrix(y_test, y_pred_nb, labels=labels_nb)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm_nb,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels_nb,
    yticklabels=labels_nb
)
plt.title("Naïve Bayes Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

In [11]:
new_emails = [
    "Congratulations! Claim your free gift certificate right now",
    "Please attend the departmental faculty meeting at 10 AM",
    "Exclusive cash prize won: click here to redeem rewards",
    "Kindly review the final submission before Friday"
]

new_email_tfidf = tfidf_spam.transform(new_emails)
new_predictions = nb_model.predict(new_email_tfidf)

new_email_results = pd.DataFrame({
    "New Email": new_emails,
    "Predicted Class": new_predictions
})

print("Predictions for Unseen Incoming Emails:")
print(new_email_results.to_string(index=False))

Predictions for Unseen Incoming Emails:
                                                  New Email Predicted Class
Congratulations! Claim your free gift certificate right now            Spam
    Please attend the departmental faculty meeting at 10 AM        Not Spam
     Exclusive cash prize won: click here to redeem rewards            Spam
           Kindly review the final submission before Friday        Not Spam


## PART 2: SENTIMENT CLASSIFICATION USING SUPPORT VECTOR MACHINE (SVM)

In [12]:
reviews = [
    # Positive (10)
    "This noise-canceling headphone provides breathtaking audio clarity",
    "The battery duration is phenomenal and lasts three full days",
    "Outstanding display sharpness and buttery smooth responsiveness",
    "Absolutely love the ergonomics and premium titanium build quality",
    "Exceptional customer support team resolved my query within minutes",
    "Fastest laptop processor I have ever experienced, stellar speed",
    "Superb camera dynamic range and crisp nighttime photos",
    "Fabulous user experience, intuitive interface, and sleek aesthetic",
    "Exceeded all my expectations in everyday productivity and gaming",
    "Top tier durability and flawless wireless connectivity",

    # Negative (10)
    "Abysmal battery drain makes this device virtually unusable",
    "The touchscreen freezes constantly and requires repeated rebooting",
    "Cheap plastic material that cracked within two weeks of light use",
    "Extremely sluggish app loading and persistent thermal throttling",
    "Horrible microphone static noise during voice calls",
    "Software is full of glitches and crashes unpredictably",
    "Terrible purchase, completely regret spending money on this unit",
    "Customer service was unhelpful and refused warranty replacement",
    "Display arrived with dead pixels and unresponsive corners",
    "Device overheats excessively during simple video playback",

    # Neutral (10)
    "The device delivers standard performance consistent with its price",
    "Average speaker loudness, neither impressive nor terrible",
    "Battery life is decent enough for everyday commuting",
    "Screen resolution is acceptable for ordinary text browsing",
    "The design is conventional and matches standard market offerings",
    "Functions as advertised without standout features",
    "Acceptable build quality for basic home usage",
    "Moderate processing power adequate for office work",
    "The device arrived on time and functions routinely",
    "Satisfactory everyday utility without noticeable flaws or highlights"
]

sentiments = ["Positive"] * 10 + ["Negative"] * 10 + ["Neutral"] * 10

review_df = pd.DataFrame({
    "review": reviews,
    "sentiment": sentiments
})

print("Dataset shape:", review_df.shape)
print("\nSentiment distribution:")
print(review_df["sentiment"].value_counts())

Dataset shape: (30, 2)

Sentiment distribution:
sentiment
Positive    10
Negative    10
Neutral     10
Name: count, dtype: int64


In [13]:
X_review_text = review_df["review"]
y_review = review_df["sentiment"]

tfidf_review = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

X_review_tfidf = tfidf_review.fit_transform(X_review_text)

print("Review TF-IDF matrix shape:", X_review_tfidf.shape)
print("Vocabulary size:", len(tfidf_review.vocabulary_))

Review TF-IDF matrix shape: (30, 310)
Vocabulary size: 310


In [14]:
X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(
    X_review_tfidf,
    y_review,
    test_size=0.3,
    random_state=42,
    stratify=y_review
)

print("Training review samples :", X_train_svm.shape[0])
print("Testing review samples  :", X_test_svm.shape[0])

Training review samples : 21
Testing review samples  : 9


In [15]:
svm_model = LinearSVC(C=1.0, random_state=42)
svm_model.fit(X_train_svm, y_train_svm)

print("Linear SVM model trained successfully.")

Linear SVM model trained successfully.


In [16]:
y_pred_svm = svm_model.predict(X_test_svm)

accuracy_svm = accuracy_score(y_test_svm, y_pred_svm)
print(f"SVM Accuracy: {accuracy_svm * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test_svm, y_pred_svm))

SVM Accuracy: 11.11%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00         3
     Neutral       0.00      0.00      0.00         3
    Positive       0.17      0.33      0.22         3

    accuracy                           0.11         9
   macro avg       0.06      0.11      0.07         9
weighted avg       0.06      0.11      0.07         9



In [17]:
labels_svm = ["Negative", "Neutral", "Positive"]
cm_svm = confusion_matrix(y_test_svm, y_pred_svm, labels=labels_svm)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_svm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=labels_svm,
    yticklabels=labels_svm
)
plt.title("SVM Sentiment Classification Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")
plt.show()

In [18]:
new_reviews = [
    "The laptop speed is breathtaking and display is gorgeous",
    "Terrible battery performance and constantly freezes",
    "The screen and weight are completely average and acceptable",
    "Exceptional sound quality, worth every penny",
    "Horrible overheating issue, completely dissatisfied"
]

new_review_tfidf = tfidf_review.transform(new_reviews)
new_predictions_svm = svm_model.predict(new_review_tfidf)

new_review_results = pd.DataFrame({
    "New Review": new_reviews,
    "Predicted Sentiment": new_predictions_svm
})

print("Predictions for Unseen Customer Reviews:")
print(new_review_results.to_string(index=False))

Predictions for Unseen Customer Reviews:
                                                 New Review Predicted Sentiment
   The laptop speed is breathtaking and display is gorgeous            Positive
        Terrible battery performance and constantly freezes            Negative
The screen and weight are completely average and acceptable             Neutral
               Exceptional sound quality, worth every penny            Positive
        Horrible overheating issue, completely dissatisfied            Positive


## PART 3 (POST-LAB): MULTI-TOPIC DOCUMENT CLASSIFICATION

In [19]:
# POST-LAB MULTI-TOPIC DATASET
news_data = {
    "text": [
        "The national cricket team won the championship by seven wickets",
        "The ministry announced fiscal tax reforms and economic stimulus",
        "Researchers unveiled next generation neural processor architecture",
        "Stock indices surged following stellar quarterly corporate earnings",
        "The director won international academy accolades for the drama film",
        "The premier league soccer club advanced to the continental finals",
        "Legislators voted to approve the comprehensive environmental bill",
        "Autonomous drone delivery networks received civil aviation approval",
        "Venture capital firms boosted funding for renewable energy ventures",
        "The upcoming fantasy cinematic adaptation released its world teaser"
    ],
    "category": [
        "Sports",
        "Politics",
        "Technology",
        "Business",
        "Entertainment",
        "Sports",
        "Politics",
        "Technology",
        "Business",
        "Entertainment"
    ]
}

df_news = pd.DataFrame(news_data)

# TF-IDF Feature Extraction
tfidf_news = TfidfVectorizer(stop_words='english')
X_news = tfidf_news.fit_transform(df_news['text'])
y_news = df_news['category']

# Train-Test Split
X_tr, X_te, y_tr, y_te = train_test_split(X_news, y_news, test_size=0.4, random_state=42)

# Train Classifier
model_news = MultinomialNB()
model_news.fit(X_tr, y_tr)

# Predict & Evaluate
y_pred_news = model_news.predict(X_te)
print("News Topic Classification Accuracy:", accuracy_score(y_te, y_pred_news))
print("\nPredictions vs Actual:")
for txt, act, pred in zip(df_news['text'].iloc[y_te.index], y_te, y_pred_news):
    print(f"Text     : {txt[:50]}...")
    print(f"Actual   : {act} | Predicted: {pred}")
    print("-" * 50)

News Topic Classification Accuracy: 0.0

Predictions vs Actual:
Text     : Venture capital firms boosted funding for renewabl...
Actual   : Business | Predicted: Technology
--------------------------------------------------
Text     : The ministry announced fiscal tax reforms and econ...
Actual   : Politics | Predicted: Technology
--------------------------------------------------
Text     : The premier league soccer club advanced to the con...
Actual   : Sports | Predicted: Technology
--------------------------------------------------
Text     : The national cricket team won the championship by ...
Actual   : Sports | Predicted: Entertainment
--------------------------------------------------

